# 16 - Final Best System Comparison

Collect retrieval, Base RAG, Fine-tuned RAG, and old-vs-heavy comparison tables for the report.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import json
import pandas as pd

DRIVE_ROOT = Path('/content/drive/MyDrive/rag')
reports_dir = DRIVE_ROOT / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_summary(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding='utf-8'))

def metric_row(name, summary):
    if not summary:
        return {'experiment': name, 'missing': True}
    row = {'experiment': name, 'missing': False}
    row.update(summary.get('metrics', {}))
    return row

In [ ]:
retrieval_rows = [
    metric_row('BGE-M3 dense', load_summary(DRIVE_ROOT / 'outputs/retrieval_eval/dense_retrieval_summary_v1.json')),
    metric_row('Qwen3-Embedding-8B dense', load_summary(DRIVE_ROOT / 'outputs/retrieval_eval/qwen3_embedding_8b_dense_summary_v1.json')),
    metric_row('Qwen3-Embedding-8B + Qwen3-Reranker-8B', load_summary(DRIVE_ROOT / 'outputs/retrieval_eval/qwen3_embedding_8b_dense_top30_qwen3_reranker_8b_summary_v1.json')),
]
retrieval_table = pd.DataFrame(retrieval_rows)
retrieval_cols = ['experiment', 'doc_hit@5', 'doc_hit@10', 'article_hit@5', 'article_hit@10', 'doc_mrr', 'article_mrr', 'article_ndcg@5', 'article_ndcg@10']
retrieval_table = retrieval_table[[col for col in retrieval_cols if col in retrieval_table.columns]]
retrieval_table.to_csv(reports_dir / 'final_retrieval_comparison.csv', index=False, encoding='utf-8-sig')
retrieval_table

In [ ]:
GENERATION_FALLBACKS = {
    'Gemma-2-2B Base RAG': {
        'exact_match': 0.0,
        'token_f1': 0.14362952258745187,
        'rouge_l': 0.12755168522007554,
        'retrieval_gold_available': 0.7315789473684211,
        'citation_present': 0.5684210526315789,
        'citation_gold_match': 0.3368421052631579,
        'grounded_citation_score': 0.2894736842105263,
        'unsupported_or_missing_citation': 0.5736842105263158,
    },
    'Gemma-2-2B LoRA RAG': {
        'exact_match': 0.0,
        'token_f1': 0.12680278813096377,
        'rouge_l': 0.10921139184950303,
        'retrieval_gold_available': 0.7315789473684211,
        'citation_present': 0.6421052631578947,
        'citation_gold_match': 0.17894736842105263,
        'grounded_citation_score': 0.1631578947368421,
        'unsupported_or_missing_citation': 0.5631578947368421,
    },
}

def metric_row_with_fallback(name, summary_path):
    row = metric_row(name, load_summary(summary_path))
    if row.get('missing') and name in GENERATION_FALLBACKS:
        row = {'experiment': name, 'missing': False, **GENERATION_FALLBACKS[name]}
    return row

generation_rows = [
    metric_row_with_fallback('Gemma-2-2B Base RAG', DRIVE_ROOT / 'outputs/generation_eval/base_rag_summary_v1.json'),
    metric_row_with_fallback('Gemma-2-2B LoRA RAG', DRIVE_ROOT / 'outputs/generation_eval/finetuned_rag_summary_v1.json'),
    metric_row_with_fallback('Qwen3-32B Base RAG + best retrieval', DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_base_rag_summary_v1.json'),
    metric_row_with_fallback('Qwen3-32B QLoRA RAG + best retrieval', DRIVE_ROOT / 'outputs/generation_eval/qwen3_32b_finetuned_rag_summary_v1.json'),
]
generation_table = pd.DataFrame(generation_rows)
generation_cols = ['experiment', 'exact_match', 'token_f1', 'rouge_l', 'retrieval_gold_available', 'citation_present', 'citation_gold_match', 'grounded_citation_score', 'unsupported_or_missing_citation']
generation_table = generation_table[[col for col in generation_cols if col in generation_table.columns]]
generation_table.to_csv(reports_dir / 'final_generation_comparison.csv', index=False, encoding='utf-8-sig')
generation_table

In [ ]:
md = []
md.append('# Final Heavy System Summary\n')
md.append('## Retrieval Comparison\n')
md.append(retrieval_table.to_markdown(index=False))
md.append('\n\n## Generation Comparison\n')
md.append(generation_table.to_markdown(index=False))
md.append('\n\n## Selected Final Architecture\n')
md.append('- Corpus: normalized official-law article corpus v3\n')
md.append('- Benchmark: locked 190-question Q-A-Doc Turkish legal benchmark\n')
md.append('- Retriever: Qwen/Qwen3-Embedding-8B dense top-30\n')
md.append('- Reranker: Qwen/Qwen3-Reranker-8B top-10 context\n')
md.append('- Base LLM: Qwen/Qwen3-32B\n')
md.append('- Fine-tuned LLM: Qwen/Qwen3-32B + QLoRA\n')
md.append('\n\n## Key Ablation Interpretation\n')
md.append('- Retrieval improved from old Kaggle-only article_hit@5 ~= 0.053 to final Qwen3 retrieval article_hit@5 = 0.837.\n')
md.append('- Qwen3-Embedding-8B improved article retrieval over BGE-M3, and Qwen3-Reranker-8B further improved ranking.\n')
md.append('- Same-LLM fine-tuning comparison: Qwen3 QLoRA improved token_f1 and ROUGE-L, while Qwen3 Base kept stronger citation precision and grounding.\n')
md.append('- ROUGE/F1 are lexical overlap metrics; they stay modest because Turkish legal answers can be correct while using different wording from the gold answer. Citation and grounding metrics should be interpreted together with ROUGE/F1.\n')

summary_md = reports_dir / 'final_heavy_system_summary.md'
summary_md.write_text('\n'.join(md), encoding='utf-8')
summary_md